# NetraGraph Model Export
Validate a generated bundle, include its report, and package it for import into the backend.

In [ ]:
from pathlib import Path
import shutil, sys
PROJECT = Path('/content/NetraGraph') if Path('/content/NetraGraph').exists() else Path.cwd()
try:
    from google.colab import drive, files
    drive.mount('/content/drive')
except ImportError:
    files = None
if not (PROJECT / 'backend/ml').exists() and files:
    uploaded = files.upload()
    source_zip = next((name for name in uploaded if name.endswith('.zip')), None)
    if source_zip:
        shutil.unpack_archive(source_zip, '/content/NetraGraph')
    PROJECT = Path('/content/NetraGraph')
if not (PROJECT / 'backend/ml').exists():
    raise FileNotFoundError('Upload a ZIP containing backend/ml for the portable trainer.')
sys.path.insert(0, str(PROJECT / 'backend'))
!pip install -q -r {PROJECT / 'requirements/requirements-colab.txt'}
BUNDLE = Path('/kaggle/working/artifacts/intrusion/v1') if Path('/kaggle/input').exists() else Path('/content/artifacts/intrusion/v1')
from ml.inference.model_loader import validate_bundle
print(validate_bundle(BUNDLE))
assert (BUNDLE / 'training_report.json').exists(), 'training_report.json must accompany the bundle'
archive = shutil.make_archive(str(BUNDLE.parent.parent / 'NetraGraph_artifact'), 'zip', root_dir=BUNDLE.parent.parent.parent, base_dir=BUNDLE.parent.parent.name)
print('Upload this ZIP to POST /api/ml/models/import:', archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass